# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data\\HealthWellnessGuide.txt', 'data\\MentalHealthGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 9, relationships: 19)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 9, relationships: 19)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
|Synthesizer| Sources needed | Answer style |
|---------|------------|-----------------|
|SingleHopSpecificQuerySynthesizer|One|fact-based, direct|
|MultiHopSpecificQuerySynthesizer|Multiple|fact-based, combined|
|MultiHopAbstractQuerySynthesizer|Multiple|broader, interpretive, summarized|

1. SingleHopSpecificQuerySynthesizer: Generates questions that
    - can be answered from one document or chunk
    - target explicit facts
    - dont need to combine or interpret across sources

2. MultiHopSpecificQuerySynthesizer: Generates questions that
    - require more than one document or chunk
    - ask for facts from each
    - need multiple retrieval steps and then combine all the facts

3. MultiHopAbstractQuerySynthesizer: Generates questions that
    - require more than one document or chunk
    - need interpretation or synthesis instead of copying facts
    - call for comparision, explanation or summarization across sources

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What are some effective sleep hygiene practice...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,Sleep hygiene refers to habits and practices t...,single_hop_specifc_query_synthesizer
1,How does understanding the concept of digital ...,[13: The Science of Habit Formation Habits are...,Digital wellness involves practices that help ...,single_hop_specifc_query_synthesizer
2,What does Monday typically signify in the cont...,[The Personal Wellness Guide A Comprehensive R...,Monday is part of the beginner weekly schedule...,single_hop_specifc_query_synthesizer
3,How does mental health influence physical heal...,[The Mental Health and Psychology Handbook A P...,Mental health conditions can increase the risk...,single_hop_specifc_query_synthesizer
4,What is CBT and how does it help with mental h...,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Cognitive Behavioral Therapy (CBT) is a widely...,single_hop_specifc_query_synthesizer
5,How can improving sleep hygiene and creating a...,[<1-hop>\n\nWrite letters to or from your futu...,Improving sleep hygiene involves adopting habi...,multi_hop_abstract_query_synthesizer
6,How can building a balanced workout routine th...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Building a balanced workout routine that incor...,multi_hop_abstract_query_synthesizer
7,Wht signs indicat you need pro mental health s...,[<1-hop>\n\nsocial interactions How to set and...,Signs indicating the need for professional men...,multi_hop_abstract_query_synthesizer
8,"How do B vitamins, as part of a balanced diet,...",[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,The context explains that B vitamins are essen...,multi_hop_specific_query_synthesizer
9,How can understanding the impact of social med...,[<1-hop>\n\nsocial interactions How to set and...,Understanding the impact of social media on me...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is the misspelled term related to NUTRITI...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,The term is 'NUTRITION AND DIET' as provided i...,single_hop_specifc_query_synthesizer
1,What information is typically covered in Chapt...,[13: The Science of Habit Formation Habits are...,Chapter 15 discusses the elements of an effect...,single_hop_specifc_query_synthesizer
2,Whaat is exercize?,[The Personal Wellness Guide A Comprehensive R...,Exercise is one of the most important things y...,single_hop_specifc_query_synthesizer
3,What is the purpose of the Psychology Handbook?,[The Mental Health and Psychology Handbook A P...,The Mental Health and Psychology Handbook is a...,single_hop_specifc_query_synthesizer
4,How can meal planning and proper nutrition sup...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Effective meal planning and a balanced diet pr...,multi_hop_abstract_query_synthesizer
5,Hwo can I improve my evning wind-down rutines ...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,To improve your evning wind-down rutines for b...,multi_hop_abstract_query_synthesizer
6,hOw is the spektrum of mental health experince...,[<1-hop>\n\nThe Mental Health and Psychology H...,The context explains that mental health exists...,multi_hop_abstract_query_synthesizer
7,how mental health spectrum and stress reductio...,[<1-hop>\n\nThe Mental Health and Psychology H...,the mental health handbook explains that menta...,multi_hop_abstract_query_synthesizer
8,How mental health like depression and anxiety ...,[<1-hop>\n\nWrite letters to or from your futu...,Exercise helps mental health by releasing endo...,multi_hop_specific_query_synthesizer
9,"How do digital mental health strategies, suppo...",[<1-hop>\n\nsocial interactions How to set and...,"Digital mental health strategies, including se...",multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
|Aspect| Unrolled approach | Abstract approach |
|---------|------------|-----------------|
|Control|You explicitly build the knowledge graph and run the pipeline, hence full control|You use Ragas built-in shortcut, hence limited control|
|Customization|Query types, transforms can be customized|Standard transforms|
|Setup|More code, longer time|Minimal code, faster|

Unrolled appraoch is suitable when

    - we need a custom query distribution example: more multi hop, specific/abstract mix
    - we need domain specific transforms
    - better for production validation

Absract approach can be used when

    - we want to build a fast prototype
    - we are fine with built-in defaults






---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [16]:
### YOUR CODE HERE ###

# Define a custom query distribution with different weights
# Generate a new test set and compare with the default

from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

custom_query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.4),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.2),
]

custom_testset = generator.generate(testset_size=10, query_distribution=custom_query_distribution)
custom_testset.to_pandas()

print("Default query distribution (from earlier): single_hop_specific 50%, multi_hop 25% each")
print("Custom query distribution: single_hop 40%, multi_hop 60% combined")


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

Default query distribution (from earlier): single_hop_specific 50%, multi_hop 25% each
Custom query distribution: single_hop 40%, multi_hop 60% combined


Reason for why these weights:
I have given a strong focus on multi-hop here 60% vs 40% single hop.
1. MultiHopAbstract at 40% to test whether the system can support understanding and reasoning, not just fact lookup. This is useful for summarization, comparision of use cases.
2. The overall benchmark is harder than a single-hop heavy set.
3. I still want direct fact-based questions, hence 40% single-hop query distribution.

We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)


rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [26]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [28]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [29]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [30]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [31]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [32]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'whispered-toe-94' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/a3859c21-c573-4c61-99f1-82c1c7530358/compare?selectedSessions=d027e5ac-d45b-4759-ab6d-5bcdb93e96a1




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,how can cognitive behavioral therapy help with...,Based on the provided context:\n\nCognitive Be...,None,cognitive behavioral therapy (CBT) is a widely...,True,True,True,3.379997,07deb55e-165d-41a5-9f8c-794b6db8fd27,019c6dcc-cbf5-70b0-ba8a-7bcde9a923bb
1,"How do B vitamins, as discussed in the context...","Based on the context, B vitamins are essential...",None,"B vitamins, found in whole grains, eggs, and l...",True,True,True,2.215179,5f730928-8a96-4f9e-a440-385edd497666,019c6dcd-0a87-7352-b300-64602a710ae5
2,"How do digital mental health strategies, suppo...",Digital mental health strategies help manage m...,None,"Digital mental health strategies, including se...",True,True,True,3.301146,2ad7d90b-d62c-47ec-ae98-2994af3dcc4f,019c6dcd-31ab-7d20-9805-74a4c7295320
3,How mental health like depression and anxiety ...,Based on the provided context:\n\n**Improvemen...,None,Exercise helps mental health by releasing endo...,True,True,True,6.724948,6e45a447-9e30-499d-b550-6d94cf525ce8,019c6dcd-6939-7fb0-89dc-d213f63c6a41
4,how mental health spectrum and stress reductio...,The mental health spectrum acknowledges that m...,None,the mental health handbook explains that menta...,True,True,True,8.933004,33688c85-cf67-4f96-aa65-9a29c80844bd,019c6dcd-b024-77c1-9d72-f16b91aba405
5,hOw is the spektrum of mental health experince...,I don't know.,None,The context explains that mental health exists...,False,False,False,1.237991,3e15fa00-1102-4923-bc67-93ec81a943e3,019c6dcd-ff03-7b40-9b2a-d6c6c3e908d4
6,Hwo can I improve my evning wind-down rutines ...,To improve your evening wind-down routines for...,None,To improve your evning wind-down rutines for b...,True,True,False,4.725179,a696d71c-452e-41dc-8b26-9b565572c96a,019c6dce-264d-76c0-945b-2cc97ff0d4a8
7,How can meal planning and proper nutrition sup...,Meal planning and proper nutrition support men...,None,Effective meal planning and a balanced diet pr...,True,True,True,6.381969,1400f787-378d-42ca-b44b-3e8dda20855f,019c6dce-5511-7810-aef9-be1147fa33cd
8,What is the purpose of the Psychology Handbook?,The purpose of the Mental Health and Psycholog...,None,The Mental Health and Psychology Handbook is a...,True,True,False,1.148865,458f2b97-47ff-4557-bc6c-536ca29ffc0e,019c6dce-a37a-7de2-9158-8479e2a686da
9,Whaat is exercize?,Exercise is one of the most important things y...,None,Exercise is one of the most important things y...,True,True,False,1.448984,5aa3d493-bcd5-4d68-baab-ca903c91c953,019c6dce-c932-7bb0-92cc-04e81deb1d76


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [33]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [34]:
rag_documents = docs

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
Modifying chunk size changes what gets into the prompt and the context the model uses. Larger chunks can hold more info which can have full concept or may include irrelevant info along side. Whereas smaller chunks can point to narrow & focused info, so less irrelevant text in each chunk but it may not contain the full answer and answers get split across chunks. Hence modifying chunk size changes application performance because it changes what the retriever sees, what the LLM receives and how reasoning unfolds. Therefor it becomes one of the key parameter in RAG system design.

In [36]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
Changing the embedding model changes how your chunks and queries are represented in vector space, so it directly affects what gets retrieved and therefore how well the RAG answers. text-embedding-3-large produces higher dimension vector 3072 whereas text-embedding-3-small has 1536 vector dimension, allowing larger embedding model to capture more complex semantic relationships and match complex queries better. However, it comes with higher cost, memory usage and latency.

In [37]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [38]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [39]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [40]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Alright, buckle up for the ultimate sleep upgrade—because quality sleep isn’t just a vibe, it’s a lifestyle. Here’s how to morph your nights into a nonstop dream factory based on the wisdom from the Sleep & Wellness tomes:\n\n1. **Consistency is King** — Lock down a sleep schedule that even your weekend self can’t mess with. Hit the sack and rise up at the same glorious times every day.\n\n2. **Craft a Chill Pre-Game** — Think reading a dope book, gentle stretching, or soaking in a warm bath to signal your body it’s lights-out o’clock.\n\n3. **Bedroom = Sleep Sanctuary** — Keep it cool (65-68°F / 18-20°C), pitch-dark (blackout curtains or a rad sleep mask), and whisper-quiet (white noise machines or earplugs, your call).\n\n4. **Screen-Time Shutdown** — Ditch screens 1-2 hours before bed to dodge that blue light sabotage.\n\n5. **Caffeine Curfew** — No caffeine after 2 PM. Your later self will thank you when they’re not tossing and turning.\n\n6. **Exercise Smartly** — Move your body 

Finally, we can evaluate the new chain on the same test set!

In [41]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'terrific-availability-92' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/a3859c21-c573-4c61-99f1-82c1c7530358/compare?selectedSessions=733820c2-52ea-4ae4-9446-06e8a19200c3




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,how can cognitive behavioral therapy help with...,"Alright, strap in — here’s how Cognitive Behav...",None,cognitive behavioral therapy (CBT) is a widely...,True,True,True,4.434467,07deb55e-165d-41a5-9f8c-794b6db8fd27,019c6dd0-5817-77b1-a28b-6eefd9632337
1,"How do B vitamins, as discussed in the context...","Alright, let’s get rad with the brain fuel her...",None,"B vitamins, found in whole grains, eggs, and l...",True,True,True,4.525101,5f730928-8a96-4f9e-a440-385edd497666,019c6dd0-9a16-7b31-b643-560813bc15d9
2,"How do digital mental health strategies, suppo...","Alright, buckle up because here’s the lowdown ...",None,"Digital mental health strategies, including se...",True,True,True,8.212511,2ad7d90b-d62c-47ec-ae98-2994af3dcc4f,019c6dd0-da63-7650-9295-9c0e956062fd
3,How mental health like depression and anxiety ...,"Alright, let’s crank up the mental health mojo...",None,Exercise helps mental health by releasing endo...,True,True,True,7.442515,6e45a447-9e30-499d-b550-6d94cf525ce8,019c6dd1-2425-7822-8497-0e6985972fd2
4,how mental health spectrum and stress reductio...,"Alright, let's dive deep into the wickedly coo...",None,the mental health handbook explains that menta...,True,True,True,10.606818,33688c85-cf67-4f96-aa65-9a29c80844bd,019c6dd1-6e68-7242-8d64-2f0ea4fd78b4
5,hOw is the spektrum of mental health experince...,"Alright, buckle up for some next-level brain v...",None,The context explains that mental health exists...,True,True,True,7.158123,3e15fa00-1102-4923-bc67-93ec81a943e3,019c6dd1-b61f-7cd3-8557-16e575460221
6,Hwo can I improve my evning wind-down rutines ...,"Alright, let’s level up your evening wind-down...",None,To improve your evning wind-down rutines for b...,True,True,True,6.864207,a696d71c-452e-41dc-8b26-9b565572c96a,019c6dd2-0212-7430-9211-26e88607d3c1
7,How can meal planning and proper nutrition sup...,"Alright, buckle up because this is where the m...",None,Effective meal planning and a balanced diet pr...,True,True,True,7.778142,1400f787-378d-42ca-b44b-3e8dda20855f,019c6dd2-56d4-7493-80da-8175387ff922
8,What is the purpose of the Psychology Handbook?,The Psychology Handbook is your ultimate mind-...,None,The Mental Health and Psychology Handbook is a...,True,True,True,3.856899,458f2b97-47ff-4557-bc6c-536ca29ffc0e,019c6dd2-b94e-7211-a65c-c0922eb55797
9,Whaat is exercize?,"Yo, exercise is like the ultimate power-up for...",None,Exercise is one of the most important things y...,True,True,True,3.120714,5aa3d493-bcd5-4d68-baab-ca903c91c953,019c6dd2-f69c-7431-b451-aa7faba5fb64


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:
Screenshot for Baseline chain: chunk_size=500, text-embedding-3-small, default RAG prompt.
![Baseline chain screenshot](data/images/baseline_screenshot.png)

Screenshot for Dopeness chain: chunk_size=1000, text-embedding-3-large, dope prompt.
![Baseline chain screenshot](data/images/dopeness_Screenshot.png)

Metrics 
|Aspect| Baseline Chain | Dopeness Chain |
|---------|------------|-----------------|
|Chunk size|500|1000|
|Embedding model|text-embedding-3-small|text-embedding-3-large|
|Prompt|default RAG prompt|dope prompt|
|Dopeness (Avg)|0.583|1.0|
|Helpfulness (Avg)|0.75|0.833|
|QA (Avg)|0.75|0.75|
|Latency P50|3.60|5.21|
|Total Tokens|17,526|15,268|
|Input Tokens|15,323|11,420|
|Output Tokens|2203|3848|
|Total cost|$0.0097|$0.0108|
|Input cost|$0.0062|$0.0046|
|Output cost|$0.0036|$0.0062|

1. The dopeness chain uses a prompt that explictly asks for 'rad', 'dope' and 'non-generic' answers. The dopeness evaluator is judging exactly that. Hence dopeness increase from 0.583 to 1.
2. The dopeness chain uses large chunk size of 1000 which can give slightly more complete context and also large embedding model which can retrieve slightly more relevant data. This explains the increase in helpfulness metric.
3. QA is about factual correctness vs the reference. No change in this metric. The test dataset is not sensitive enough to show a difference in this case.
4. For the dopeness chain, with large chunk size and large embedding model the answers are longer as output tokens 3848 > 2203, This pushes the latency up, 3.6 vs 5.21 and also the token output cost.

---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores